# EoH Baseline Run (HPC vLLM)

This notebook is for the original EoH baseline (`baseline/original-eoh`).
It starts a local `/completions` bridge that forwards to your internal OpenAI-compatible vLLM endpoint.


In [ ]:
import os, subprocess, eoh
print('eoh package:', eoh.__file__)
print('cwd:', os.getcwd())
print('branch:', subprocess.check_output(['git','rev-parse','--abbrev-ref','HEAD'], text=True).strip())
print('last commit:', subprocess.check_output(['git','log','-1','--oneline'], text=True).strip())


## Configure Endpoint
Set your API key in an environment variable before running this cell:

`export ENSIA_VLLM_API_KEY='...'` (terminal) or `os.environ['ENSIA_VLLM_API_KEY']='...'` (notebook).


In [ ]:
import os

VLLM_BASE = os.getenv('ENSIA_VLLM_BASE', 'http://vllm-nodeport.vllm-ns.svc.cluster.local:8000/v1')
MODEL = os.getenv('ENSIA_VLLM_MODEL', 'QuantTrio/Qwen3-VL-235B-A22B-Instruct-AWQ')
API_KEY = os.getenv('ENSIA_VLLM_API_KEY')
PORT = int(os.getenv('EOH_BRIDGE_PORT', '18000'))

if not API_KEY:
    raise RuntimeError('Missing ENSIA_VLLM_API_KEY environment variable.')

print('VLLM_BASE:', VLLM_BASE)
print('MODEL:', MODEL)
print('PORT:', PORT)


In [ ]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
import requests

class BridgeHandler(BaseHTTPRequestHandler):
    def _send_json(self, code, payload):
        body = json.dumps(payload).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_POST(self):
        if self.path != '/completions':
            self._send_json(404, {'error': 'not found'})
            return
        try:
            length = int(self.headers.get('Content-Length', '0'))
            req = json.loads(self.rfile.read(length).decode('utf-8'))
            prompt = req.get('prompt', '')

            payload = {
                'model': MODEL,
                'messages': [{'role': 'user', 'content': prompt}],
                'temperature': 0.2,
            }
            headers = {
                'Authorization': f'Bearer {API_KEY}',
                'Content-Type': 'application/json',
            }
            r = requests.post(f'{VLLM_BASE}/chat/completions', headers=headers, json=payload, timeout=600)
            if r.status_code != 200:
                self._send_json(502, {'error': 'upstream', 'status': r.status_code, 'text': r.text[:500]})
                return

            data = r.json()
            text = data['choices'][0]['message']['content']
            self._send_json(200, {'content': [text]})
        except Exception as e:
            self._send_json(500, {'error': str(e)})

server = HTTPServer(('127.0.0.1', PORT), BridgeHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print(f'Bridge running at http://127.0.0.1:{PORT}/completions')


In [ ]:
import requests
test = {
    'prompt': 'Reply with exactly: OK',
    'repeat_prompt': 1,
    'params': {'do_sample': True}
}
r = requests.post(f'http://127.0.0.1:{PORT}/completions', json=test, timeout=180)
print(r.status_code)
print(r.json())


In [ ]:
from eoh import eoh
from eoh.utils.getParas import Paras

paras = Paras()
paras.set_paras(
    method='eoh',
    problem='tsp_construct',
    llm_use_local=True,
    llm_local_url=f'http://127.0.0.1:{PORT}/completions',
    llm_model=MODEL,
    ec_pop_size=2,
    ec_n_pop=2,
    exp_n_proc=1,
    exp_debug_mode=False,
)

evolution = eoh.EVOL(paras)
evolution.run()


In [ ]:
from eoh import eoh
from eoh.utils.getParas import Paras

paras = Paras()
paras.set_paras(
    method='eoh',
    problem='bp_online',
    llm_use_local=True,
    llm_local_url=f'http://127.0.0.1:{PORT}/completions',
    llm_model=MODEL,
    ec_pop_size=2,
    ec_n_pop=2,
    exp_n_proc=1,
    exp_debug_mode=False,
)

evolution = eoh.EVOL(paras)
evolution.run()


In [ ]:
import os, glob
print('results exists:', os.path.isdir('./results'))
print('pops:', glob.glob('./results/pops/population_generation_*.json'))
print('best:', glob.glob('./results/pops_best/population_generation_*.json'))
